DOMAIN SPECIFIC PROMPT EVALUATION METRIC

BATTERY HOLDER PROMPT EVALUATION

In [9]:
# ---- Install dependencies (run once)
# !pip install torch transformers textstat pandas numpy tqdm sentencepiece language-tool-python sentence-transformers scikit-learn openpyxl

import os
import re
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import textstat
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import language_tool_python
from sentence_transformers import SentenceTransformer

# ============================================================
# ---------------- Configuration -----------------------------
# ============================================================

CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "lm_model_name": "gpt2",
    "max_length": 1024,
    "stride": 512,
    "coherence_model": "sentence-transformers/all-MiniLM-L6-v2",
    "input_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\06_BATTERY_HOLDER_PROMPTS.csv",
    "output_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\06_BATTERY_HOLDER_PROMPTS_metrics.csv",
    "output_excel": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\06_BATTERY_HOLDER_PROMPTS_metrics.xlsx",

    # 👇 Correct public API host (no trailing slash needed, but allowed)
    "lt_api_url": "https://api.languagetool.org/v2",
    "lt_max_retries": 5,
    "lt_retry_delay": 1.0,
    "lt_request_delay": 0.2,  # ~5 req/sec — safer for public API
}

# Speed up CPU runs a bit
if CONFIG["device"] == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() - 1))

# ============================================================
# ---------------- Utility Functions -------------------------
# ============================================================

_word_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")

def simple_word_tokenize(text: str):
    return _word_re.findall(text.lower())

def sentence_split(text: str):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sents if s.strip()]

def ttr(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def mtld(tokens, ttr_threshold=0.72, min_segment=10):
    def _mtld_calc(seq):
        factors, types, token_count = 0, set(), 0
        for tok in seq:
            token_count += 1
            types.add(tok)
            if token_count >= min_segment:
                if (len(types) / token_count) <= ttr_threshold:
                    factors += 1
                    types, token_count = set(), 0
        if token_count != 0:
            # handle final partial factor
            denom = max(1 - ttr_threshold, 1e-8)
            factors += (1 - ((len(types) / max(1, token_count)) - ttr_threshold) / denom)
        return len(seq) / max(factors, 1e-8)

    if len(tokens) < min_segment:
        return float("nan")
    forward = _mtld_calc(tokens)
    reverse = _mtld_calc(list(reversed(tokens)))
    return (forward + reverse) / 2.0

@torch.no_grad()
def perplexity(text, model, tokenizer, max_length=1024, stride=512, device="cpu"):
    if not text.strip():
        return float("nan")
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = enc.input_ids.to(device)
    nlls, seq_len = [], input_ids.size(1)
    for i in range(0, seq_len, stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, seq_len)
        trg_len = end_loc - i
        input_ids_slice = input_ids[:, begin_loc:end_loc]
        target_ids = input_ids_slice.clone()
        target_ids[:, :-trg_len] = -100
        outputs = model(input_ids=input_ids_slice, labels=target_ids)
        nlls.append(outputs.loss * trg_len)
        if end_loc == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / seq_len).item()
    return float(ppl)

def safe_grammar_check(text, tool, max_retries=5, retry_delay=1.0, request_delay=0.2):
    """Returns (num_errors, num_tokens, error_rate). If tool is None or fails, returns NaNs gracefully."""
    tokens = simple_word_tokenize(text) if isinstance(text, str) else []
    if not tokens or tool is None:
        return (0, len(tokens), float("nan"))

    for attempt in range(max_retries):
        try:
            time.sleep(request_delay)  # be nice to the public API
            matches = tool.check(text)
            num_errors = sum(1 for m in matches if m.ruleId != "WHITESPACE_RULE")
            ger = num_errors / len(tokens) if len(tokens) else float("nan")
            return (num_errors, len(tokens), ger)
        except language_tool_python.RateLimitError:
            backoff = retry_delay * (attempt + 1)
            print(f"⚠️ LanguageTool rate-limited. Retrying in {backoff:.1f}s...")
            time.sleep(backoff)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Grammar check failed after {max_retries} tries: {e}")
                return (0, len(tokens), float("nan"))
            time.sleep(retry_delay * (attempt + 1))
    return (0, len(tokens), float("nan"))

def coherence_adjacent_cosine(text, embedder):
    sents = sentence_split(text)
    sents = [s for s in sents if simple_word_tokenize(s)]
    if len(sents) < 2:
        return float("nan")
    embs = embedder.encode(sents, normalize_embeddings=True, show_progress_bar=False)
    sims = [(embs[i] @ embs[i+1]) for i in range(len(embs)-1)]
    return float(np.mean(sims)) if sims else float("nan")

# ============================================================
# --------------------- Main Pipeline ------------------------
# ============================================================

print(f"Using device: {CONFIG['device']}")

# 1) Load language + coherence models
tokenizer = AutoTokenizer.from_pretrained(CONFIG["lm_model_name"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["lm_model_name"]).to(CONFIG["device"]).eval()

# LanguageTool: use correct public API host; fall back to None if it fails
lt_tool = None
try:
    lt_tool = language_tool_python.LanguageTool('en-US', remote_server=CONFIG["lt_api_url"])
    # quick ping to lock in language list (avoids late failure)
    _ = lt_tool._get_languages()
    print("✅ LanguageTool connected via public API.")
except Exception as e:
    lt_tool = None
    print(f"⚠️ LanguageTool disabled (API not reachable): {e}")

embedder = SentenceTransformer(CONFIG["coherence_model"])

# 2) Load prompts
if not os.path.exists(CONFIG["input_csv"]):
    raise FileNotFoundError(f"Input CSV not found: {CONFIG['input_csv']}")

df_in = pd.read_csv(CONFIG["input_csv"])
if "prompt" not in df_in.columns:
    raise ValueError("CSV must contain a 'prompt' column")

prompts = df_in["prompt"].dropna().astype(str).tolist()
print(f"✅ Loaded {len(prompts)} prompts.")

# 3) Score prompts
rows = []
for idx, prompt in enumerate(tqdm(prompts, desc="Scoring prompts")):
    tokens = simple_word_tokenize(prompt)
    num_chars = len(prompt)
    num_words = len(tokens)
    num_sents = len(sentence_split(prompt))

    ppl = perplexity(prompt, model, tokenizer, CONFIG["max_length"], CONFIG["stride"], CONFIG["device"])
    fre = textstat.flesch_reading_ease(prompt)
    fkgl = textstat.flesch_kincaid_grade(prompt)
    fog = textstat.gunning_fog(prompt)
    ttr_score = ttr(tokens)
    mtld_score = mtld(tokens)
    err_count, tok_count, ger = safe_grammar_check(
        prompt, lt_tool,
        max_retries=CONFIG["lt_max_retries"],
        retry_delay=CONFIG["lt_retry_delay"],
        request_delay=CONFIG["lt_request_delay"]
    )
    coh = coherence_adjacent_cosine(prompt, embedder)

    rows.append({
        "id": idx,
        "prompt": prompt,
        "chars": num_chars,
        "words": num_words,
        "sentences": num_sents,
        "perplexity": round(ppl, 4) if not pd.isna(ppl) else np.nan,
        "readability_fre": round(fre, 2),
        "readability_fkgl": round(fkgl, 2),
        "readability_fog": round(fog, 2),
        "lexdiv_ttr": round(ttr_score, 4),
        "lexdiv_mtld": round(mtld_score, 4) if not pd.isna(mtld_score) else np.nan,
        "grammar_errors": err_count,
        "grammar_tokens": tok_count,
        "grammar_error_rate": round(ger, 4) if not pd.isna(ger) else np.nan,
        "coherence_adjacent_cos": round(coh, 4) if not pd.isna(coh) else np.nan
    })

# 4) Save results
df_out = pd.DataFrame(rows)

# Jupyter-friendly display (won't crash in scripts)
try:
    pd.set_option("display.max_colwidth", 120)
    pd.set_option("display.float_format", "{:.4f}".format)
    from IPython.display import display
    display(df_out.head(10))
except Exception:
    pass

df_out.to_csv(CONFIG["output_csv"], index=False, encoding='utf-8')
df_out.to_excel(CONFIG["output_excel"], index=False, engine='openpyxl')

print("\nResults saved to:")
print(f"   - CSV:   {CONFIG['output_csv']}")
print(f"   - Excel: {CONFIG['output_excel']}")


Using device: cpu
✅ LanguageTool connected via public API.
✅ Loaded 20 prompts.


Scoring prompts: 100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


,id,prompt,chars,words,sentences,perplexity,readability_fre,readability_fkgl,readability_fog,lexdiv_ttr,lexdiv_mtld,grammar_errors,grammar_tokens,grammar_error_rate,coherence_adjacent_cos
0,0,Design a 3D-printable AA battery holder with dimensions 60×30×15 mm. The holder should have a wall thickness of 2 mm...,432,85,4,25.5802,71.8800,6.0000,9.5900,0.6118,52.3599,0,85,0.0000,0.3783
1,1,Create a 3D model for a CR2032 coin cell battery holder. The external dimensions should be 25×25×8 mm with a wall th...,408,85,5,26.2142,79.7200,4.9400,8.0600,0.6235,38.1809,0,85,0.0000,0.3004
2,2,Design an enclosure for a 9V battery with dimensions 50×30×20 mm. The enclosure should have a wall thickness of 2 mm...,413,78,5,26.2678,67.6100,6.4200,10.7500,0.6154,41.2075,0,78,0.0000,0.3681
3,3,Develop a 3D model for a AAA battery holder to fit four batteries. The external dimensions should be 50×20×15 mm wit...,399,78,5,34.5965,67.2800,6.1100,10.8200,0.6795,59.5824,1,78,0.0128,0.2415
4,4,Create a 3D-printable holder for a single D-size battery. The holder should measure 70×34×34 mm with a wall thicknes...,360,71,5,25.9703,72.6200,5.8600,8.6400,0.6761,50.9841,0,71,0.0000,0.3155
5,5,Design a 3D model for a double A battery holder with dimensions 55×30×15 mm. The holder should have a wall thickness...,379,78,4,35.0132,76.3500,6.2900,8.6200,0.6410,46.2408,0,78,0.0000,0.4595
6,6,Develop a 3D-printable design for a 18650 battery holder with dimensions 80×22×22 mm. The holder should have a wall ...,365,73,4,21.2343,76.7700,4.8900,8.6200,0.6712,50.0056,0,73,0.0000,0.2746
7,7,"Create a model for a 3D-printable button cell battery holder, designed to fit LR44 batteries. The holder should have...",393,77,5,29.4088,72.7700,5.0100,8.2800,0.6623,53.3049,0,77,0.0000,0.3291
8,8,Design an enclosure for a 3.7V LiPo battery with external dimensions of 60×40×10 mm. The enclosure should have a wal...,373,74,4,19.1324,74.2100,5.2500,9.8300,0.6351,53.0138,1,74,0.0135,0.4306
9,9,"Create a 3D-printable design for a C-size battery holder, measuring 60×30×30 mm with a wall thickness of 2 mm. The c...",341,68,4,21.8766,71.8500,6.3700,10.1200,0.6618,46.2842,0,68,0.0000,0.3326



✅ Results saved to:
   - CSV:   G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\06_BATTERY_HOLDER_PROMPTS_metrics.csv
   - Excel: G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\06_BATTERY_HOLDER_PROMPTS_metrics.xlsx


ARDUINO CASE PROMPT EVALUATION

In [10]:
# ---- Install dependencies (run once)
# !pip install torch transformers textstat pandas numpy tqdm sentencepiece language-tool-python sentence-transformers scikit-learn openpyxl

import os
import re
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import textstat
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import language_tool_python
from sentence_transformers import SentenceTransformer

# ============================================================
# ---------------- Configuration -----------------------------
# ============================================================

CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "lm_model_name": "gpt2",
    "max_length": 1024,
    "stride": 512,
    "coherence_model": "sentence-transformers/all-MiniLM-L6-v2",
    "input_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\05_ARDUINO_CASE_PROMPT.csv",
    "output_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\05_ARDUINO_CASE_PROMPT_metrics.csv",
    "output_excel": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\05_ARDUINO_CASE_PROMPT_metrics.xlsx",

    # 👇 Correct public API host (no trailing slash needed, but allowed)
    "lt_api_url": "https://api.languagetool.org/v2",
    "lt_max_retries": 5,
    "lt_retry_delay": 1.0,
    "lt_request_delay": 0.2,  # ~5 req/sec — safer for public API
}

# Speed up CPU runs a bit
if CONFIG["device"] == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() - 1))

# ============================================================
# ---------------- Utility Functions -------------------------
# ============================================================

_word_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")

def simple_word_tokenize(text: str):
    return _word_re.findall(text.lower())

def sentence_split(text: str):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sents if s.strip()]

def ttr(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def mtld(tokens, ttr_threshold=0.72, min_segment=10):
    def _mtld_calc(seq):
        factors, types, token_count = 0, set(), 0
        for tok in seq:
            token_count += 1
            types.add(tok)
            if token_count >= min_segment:
                if (len(types) / token_count) <= ttr_threshold:
                    factors += 1
                    types, token_count = set(), 0
        if token_count != 0:
            # handle final partial factor
            denom = max(1 - ttr_threshold, 1e-8)
            factors += (1 - ((len(types) / max(1, token_count)) - ttr_threshold) / denom)
        return len(seq) / max(factors, 1e-8)

    if len(tokens) < min_segment:
        return float("nan")
    forward = _mtld_calc(tokens)
    reverse = _mtld_calc(list(reversed(tokens)))
    return (forward + reverse) / 2.0

@torch.no_grad()
def perplexity(text, model, tokenizer, max_length=1024, stride=512, device="cpu"):
    if not text.strip():
        return float("nan")
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = enc.input_ids.to(device)
    nlls, seq_len = [], input_ids.size(1)
    for i in range(0, seq_len, stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, seq_len)
        trg_len = end_loc - i
        input_ids_slice = input_ids[:, begin_loc:end_loc]
        target_ids = input_ids_slice.clone()
        target_ids[:, :-trg_len] = -100
        outputs = model(input_ids=input_ids_slice, labels=target_ids)
        nlls.append(outputs.loss * trg_len)
        if end_loc == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / seq_len).item()
    return float(ppl)

def safe_grammar_check(text, tool, max_retries=5, retry_delay=1.0, request_delay=0.2):
    """Returns (num_errors, num_tokens, error_rate). If tool is None or fails, returns NaNs gracefully."""
    tokens = simple_word_tokenize(text) if isinstance(text, str) else []
    if not tokens or tool is None:
        return (0, len(tokens), float("nan"))

    for attempt in range(max_retries):
        try:
            time.sleep(request_delay)  # be nice to the public API
            matches = tool.check(text)
            num_errors = sum(1 for m in matches if m.ruleId != "WHITESPACE_RULE")
            ger = num_errors / len(tokens) if len(tokens) else float("nan")
            return (num_errors, len(tokens), ger)
        except language_tool_python.RateLimitError:
            backoff = retry_delay * (attempt + 1)
            print(f"⚠️ LanguageTool rate-limited. Retrying in {backoff:.1f}s...")
            time.sleep(backoff)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Grammar check failed after {max_retries} tries: {e}")
                return (0, len(tokens), float("nan"))
            time.sleep(retry_delay * (attempt + 1))
    return (0, len(tokens), float("nan"))

def coherence_adjacent_cosine(text, embedder):
    sents = sentence_split(text)
    sents = [s for s in sents if simple_word_tokenize(s)]
    if len(sents) < 2:
        return float("nan")
    embs = embedder.encode(sents, normalize_embeddings=True, show_progress_bar=False)
    sims = [(embs[i] @ embs[i+1]) for i in range(len(embs)-1)]
    return float(np.mean(sims)) if sims else float("nan")

# ============================================================
# --------------------- Main Pipeline ------------------------
# ============================================================

print(f"Using device: {CONFIG['device']}")

# 1) Load language + coherence models
tokenizer = AutoTokenizer.from_pretrained(CONFIG["lm_model_name"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["lm_model_name"]).to(CONFIG["device"]).eval()

# LanguageTool: use correct public API host; fall back to None if it fails
lt_tool = None
try:
    lt_tool = language_tool_python.LanguageTool('en-US', remote_server=CONFIG["lt_api_url"])
    # quick ping to lock in language list (avoids late failure)
    _ = lt_tool._get_languages()
    print("✅ LanguageTool connected via public API.")
except Exception as e:
    lt_tool = None
    print(f"⚠️ LanguageTool disabled (API not reachable): {e}")

embedder = SentenceTransformer(CONFIG["coherence_model"])

# 2) Load prompts
if not os.path.exists(CONFIG["input_csv"]):
    raise FileNotFoundError(f"Input CSV not found: {CONFIG['input_csv']}")

df_in = pd.read_csv(CONFIG["input_csv"])
if "prompt" not in df_in.columns:
    raise ValueError("CSV must contain a 'prompt' column")

prompts = df_in["prompt"].dropna().astype(str).tolist()
print(f"✅ Loaded {len(prompts)} prompts.")

# 3) Score prompts
rows = []
for idx, prompt in enumerate(tqdm(prompts, desc="Scoring prompts")):
    tokens = simple_word_tokenize(prompt)
    num_chars = len(prompt)
    num_words = len(tokens)
    num_sents = len(sentence_split(prompt))

    ppl = perplexity(prompt, model, tokenizer, CONFIG["max_length"], CONFIG["stride"], CONFIG["device"])
    fre = textstat.flesch_reading_ease(prompt)
    fkgl = textstat.flesch_kincaid_grade(prompt)
    fog = textstat.gunning_fog(prompt)
    ttr_score = ttr(tokens)
    mtld_score = mtld(tokens)
    err_count, tok_count, ger = safe_grammar_check(
        prompt, lt_tool,
        max_retries=CONFIG["lt_max_retries"],
        retry_delay=CONFIG["lt_retry_delay"],
        request_delay=CONFIG["lt_request_delay"]
    )
    coh = coherence_adjacent_cosine(prompt, embedder)

    rows.append({
        "id": idx,
        "prompt": prompt,
        "chars": num_chars,
        "words": num_words,
        "sentences": num_sents,
        "perplexity": round(ppl, 4) if not pd.isna(ppl) else np.nan,
        "readability_fre": round(fre, 2),
        "readability_fkgl": round(fkgl, 2),
        "readability_fog": round(fog, 2),
        "lexdiv_ttr": round(ttr_score, 4),
        "lexdiv_mtld": round(mtld_score, 4) if not pd.isna(mtld_score) else np.nan,
        "grammar_errors": err_count,
        "grammar_tokens": tok_count,
        "grammar_error_rate": round(ger, 4) if not pd.isna(ger) else np.nan,
        "coherence_adjacent_cos": round(coh, 4) if not pd.isna(coh) else np.nan
    })

# 4) Save results
df_out = pd.DataFrame(rows)

# Jupyter-friendly display (won't crash in scripts)
try:
    pd.set_option("display.max_colwidth", 120)
    pd.set_option("display.float_format", "{:.4f}".format)
    from IPython.display import display
    display(df_out.head(10))
except Exception:
    pass

df_out.to_csv(CONFIG["output_csv"], index=False, encoding='utf-8')
df_out.to_excel(CONFIG["output_excel"], index=False, engine='openpyxl')

print("\nResults saved to:")
print(f"   - CSV:   {CONFIG['output_csv']}")
print(f"   - Excel: {CONFIG['output_excel']}")


Using device: cpu
✅ LanguageTool connected via public API.
✅ Loaded 20 prompts.


Scoring prompts: 100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


,id,prompt,chars,words,sentences,perplexity,readability_fre,readability_fkgl,readability_fog,lexdiv_ttr,lexdiv_mtld,grammar_errors,grammar_tokens,grammar_error_rate,coherence_adjacent_cos
0,0,Design an Arduino Uno case with external dimensions of 110×70×25 mm. The enclosure should have a wall thickness of 2...,445,87,4,31.6844,75.6700,6.7800,9.9700,0.6667,68.6576,1,87,0.0115,0.3071
1,1,Create an enclosure for Arduino Uno measuring 120×80×30 mm. The box should be 3 mm thick with a removable lid secure...,369,72,4,68.6943,68.9000,7.8700,10.3300,0.8056,103.6800,1,72,0.0139,0.2772
2,2,"Design a compact Arduino Nano case with dimensions of 60×25×15 mm. The case should feature a 1.5 mm wall thickness, ...",372,76,4,32.9340,81.6300,4.8100,6.9900,0.6974,65.6175,0,76,0.0000,0.2334
3,3,Develop a robust case for Arduino Mega with exterior dimensions of 125×95×35 mm. The enclosure should have 2.5 mm th...,375,74,4,46.7288,73.0200,5.8500,9.2400,0.7432,80.6989,0,74,0.0000,0.2678
4,4,Create an enclosure for an Arduino Proto Shield with dimensions of 100×50×20 mm. The walls should be 2 mm thick with...,368,78,4,34.2576,83.2000,5.3300,8.0800,0.6538,58.6376,0,78,0.0000,0.3013
5,5,Design a versatile Arduino Uno enclosure with overall dimensions of 130×75×25 mm. The case should be 2.2 mm thick an...,353,70,4,39.2304,74.8000,5.5200,8.6400,0.7571,80.7059,1,70,0.0143,0.3605
6,6,Develop a slim Arduino Nano case with dimensions 65×30×20 mm. The design should feature a 1.8 mm wall thickness and ...,330,66,4,45.6792,82.7900,4.2400,6.6900,0.7576,76.2300,1,66,0.0152,0.3432
7,7,Construct an Arduino Mega enclosure with measurements of 135×100×40 mm. The case should be 3 mm thick with a detacha...,371,71,4,61.2197,74.5400,6.1900,9.5400,0.7746,88.2175,0,71,0.0000,0.2928
8,8,"Design a compact enclosure for an Arduino Proto Shield, measuring 95×45×18 mm. The walls should be 2 mm thick with a...",355,69,4,42.6073,74.8000,5.5200,8.6400,0.7246,69.0000,0,69,0.0000,0.3218
9,9,Create an Arduino Uno case with dimensions of 115×65×30 mm. The enclosure should have 2 mm thick walls and a slide-o...,334,67,4,43.7491,83.9300,4.6800,6.9400,0.7463,73.9365,1,67,0.0149,0.3572



Results saved to:
   - CSV:   G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\05_ARDUINO_CASE_PROMPT_metrics.csv
   - Excel: G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\05_ARDUINO_CASE_PROMPT_metrics.xlsx


SENSOR HOUSING PROMPT EVALUATION

In [11]:
# ---- Install dependencies (run once)
# !pip install torch transformers textstat pandas numpy tqdm sentencepiece language-tool-python sentence-transformers scikit-learn openpyxl

import os
import re
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import textstat
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import language_tool_python
from sentence_transformers import SentenceTransformer

# ============================================================
# ---------------- Configuration -----------------------------
# ============================================================

CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "lm_model_name": "gpt2",
    "max_length": 1024,
    "stride": 512,
    "coherence_model": "sentence-transformers/all-MiniLM-L6-v2",
    "input_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\07_SENSOR_HOUING_PROMPT.csv",
    "output_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\07_SENSOR_HOUING_PROMPT_metrics.csv",
    "output_excel": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\07_SENSOR_HOUING_PROMPT_metrics.xlsx",

    # 👇 Correct public API host (no trailing slash needed, but allowed)
    "lt_api_url": "https://api.languagetool.org/v2",
    "lt_max_retries": 5,
    "lt_retry_delay": 1.0,
    "lt_request_delay": 0.2,  # ~5 req/sec — safer for public API
}

# Speed up CPU runs a bit
if CONFIG["device"] == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() - 1))

# ============================================================
# ---------------- Utility Functions -------------------------
# ============================================================

_word_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")

def simple_word_tokenize(text: str):
    return _word_re.findall(text.lower())

def sentence_split(text: str):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sents if s.strip()]

def ttr(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def mtld(tokens, ttr_threshold=0.72, min_segment=10):
    def _mtld_calc(seq):
        factors, types, token_count = 0, set(), 0
        for tok in seq:
            token_count += 1
            types.add(tok)
            if token_count >= min_segment:
                if (len(types) / token_count) <= ttr_threshold:
                    factors += 1
                    types, token_count = set(), 0
        if token_count != 0:
            # handle final partial factor
            denom = max(1 - ttr_threshold, 1e-8)
            factors += (1 - ((len(types) / max(1, token_count)) - ttr_threshold) / denom)
        return len(seq) / max(factors, 1e-8)

    if len(tokens) < min_segment:
        return float("nan")
    forward = _mtld_calc(tokens)
    reverse = _mtld_calc(list(reversed(tokens)))
    return (forward + reverse) / 2.0

@torch.no_grad()
def perplexity(text, model, tokenizer, max_length=1024, stride=512, device="cpu"):
    if not text.strip():
        return float("nan")
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = enc.input_ids.to(device)
    nlls, seq_len = [], input_ids.size(1)
    for i in range(0, seq_len, stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, seq_len)
        trg_len = end_loc - i
        input_ids_slice = input_ids[:, begin_loc:end_loc]
        target_ids = input_ids_slice.clone()
        target_ids[:, :-trg_len] = -100
        outputs = model(input_ids=input_ids_slice, labels=target_ids)
        nlls.append(outputs.loss * trg_len)
        if end_loc == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / seq_len).item()
    return float(ppl)

def safe_grammar_check(text, tool, max_retries=5, retry_delay=1.0, request_delay=0.2):
    """Returns (num_errors, num_tokens, error_rate). If tool is None or fails, returns NaNs gracefully."""
    tokens = simple_word_tokenize(text) if isinstance(text, str) else []
    if not tokens or tool is None:
        return (0, len(tokens), float("nan"))

    for attempt in range(max_retries):
        try:
            time.sleep(request_delay)  # be nice to the public API
            matches = tool.check(text)
            num_errors = sum(1 for m in matches if m.ruleId != "WHITESPACE_RULE")
            ger = num_errors / len(tokens) if len(tokens) else float("nan")
            return (num_errors, len(tokens), ger)
        except language_tool_python.RateLimitError:
            backoff = retry_delay * (attempt + 1)
            print(f"⚠️ LanguageTool rate-limited. Retrying in {backoff:.1f}s...")
            time.sleep(backoff)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Grammar check failed after {max_retries} tries: {e}")
                return (0, len(tokens), float("nan"))
            time.sleep(retry_delay * (attempt + 1))
    return (0, len(tokens), float("nan"))

def coherence_adjacent_cosine(text, embedder):
    sents = sentence_split(text)
    sents = [s for s in sents if simple_word_tokenize(s)]
    if len(sents) < 2:
        return float("nan")
    embs = embedder.encode(sents, normalize_embeddings=True, show_progress_bar=False)
    sims = [(embs[i] @ embs[i+1]) for i in range(len(embs)-1)]
    return float(np.mean(sims)) if sims else float("nan")

# ============================================================
# --------------------- Main Pipeline ------------------------
# ============================================================

print(f"Using device: {CONFIG['device']}")

# 1) Load language + coherence models
tokenizer = AutoTokenizer.from_pretrained(CONFIG["lm_model_name"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["lm_model_name"]).to(CONFIG["device"]).eval()

# LanguageTool: use correct public API host; fall back to None if it fails
lt_tool = None
try:
    lt_tool = language_tool_python.LanguageTool('en-US', remote_server=CONFIG["lt_api_url"])
    # quick ping to lock in language list (avoids late failure)
    _ = lt_tool._get_languages()
    print("✅ LanguageTool connected via public API.")
except Exception as e:
    lt_tool = None
    print(f"⚠️ LanguageTool disabled (API not reachable): {e}")

embedder = SentenceTransformer(CONFIG["coherence_model"])

# 2) Load prompts
if not os.path.exists(CONFIG["input_csv"]):
    raise FileNotFoundError(f"Input CSV not found: {CONFIG['input_csv']}")

df_in = pd.read_csv(CONFIG["input_csv"])
if "prompt" not in df_in.columns:
    raise ValueError("CSV must contain a 'prompt' column")

prompts = df_in["prompt"].dropna().astype(str).tolist()
print(f"✅ Loaded {len(prompts)} prompts.")

# 3) Score prompts
rows = []
for idx, prompt in enumerate(tqdm(prompts, desc="Scoring prompts")):
    tokens = simple_word_tokenize(prompt)
    num_chars = len(prompt)
    num_words = len(tokens)
    num_sents = len(sentence_split(prompt))

    ppl = perplexity(prompt, model, tokenizer, CONFIG["max_length"], CONFIG["stride"], CONFIG["device"])
    fre = textstat.flesch_reading_ease(prompt)
    fkgl = textstat.flesch_kincaid_grade(prompt)
    fog = textstat.gunning_fog(prompt)
    ttr_score = ttr(tokens)
    mtld_score = mtld(tokens)
    err_count, tok_count, ger = safe_grammar_check(
        prompt, lt_tool,
        max_retries=CONFIG["lt_max_retries"],
        retry_delay=CONFIG["lt_retry_delay"],
        request_delay=CONFIG["lt_request_delay"]
    )
    coh = coherence_adjacent_cosine(prompt, embedder)

    rows.append({
        "id": idx,
        "prompt": prompt,
        "chars": num_chars,
        "words": num_words,
        "sentences": num_sents,
        "perplexity": round(ppl, 4) if not pd.isna(ppl) else np.nan,
        "readability_fre": round(fre, 2),
        "readability_fkgl": round(fkgl, 2),
        "readability_fog": round(fog, 2),
        "lexdiv_ttr": round(ttr_score, 4),
        "lexdiv_mtld": round(mtld_score, 4) if not pd.isna(mtld_score) else np.nan,
        "grammar_errors": err_count,
        "grammar_tokens": tok_count,
        "grammar_error_rate": round(ger, 4) if not pd.isna(ger) else np.nan,
        "coherence_adjacent_cos": round(coh, 4) if not pd.isna(coh) else np.nan
    })

# 4) Save results
df_out = pd.DataFrame(rows)

# Jupyter-friendly display (won't crash in scripts)
try:
    pd.set_option("display.max_colwidth", 120)
    pd.set_option("display.float_format", "{:.4f}".format)
    from IPython.display import display
    display(df_out.head(10))
except Exception:
    pass

df_out.to_csv(CONFIG["output_csv"], index=False, encoding='utf-8')
df_out.to_excel(CONFIG["output_excel"], index=False, engine='openpyxl')

print("\nResults saved to:")
print(f"   - CSV:   {CONFIG['output_csv']}")
print(f"   - Excel: {CONFIG['output_excel']}")


Using device: cpu
✅ LanguageTool connected via public API.
✅ Loaded 20 prompts.


Scoring prompts: 100%|██████████| 20/20 [00:19<00:00,  1.04it/s]


,id,prompt,chars,words,sentences,perplexity,readability_fre,readability_fkgl,readability_fog,lexdiv_ttr,lexdiv_mtld,grammar_errors,grammar_tokens,grammar_error_rate,coherence_adjacent_cos
0,0,Design a sensor housing with external dimensions of 100×60×30 mm. The enclosure should have a wall thickness of 2 mm...,474,90,4,31.7453,63.3700,9.8900,13.3500,0.6556,55.4840,0,90,0.0000,0.3365
1,1,Create a compact sensor housing with dimensions 80×50×20 mm and a wall thickness of 1.5 mm. This design should inclu...,392,73,4,36.0172,59.7600,9.2700,12.1400,0.7534,82.8956,0,73,0.0000,0.2234
2,2,Design an IP65-rated sensor housing with overall dimensions of 120×80×40 mm and a wall thickness of 3 mm. Include a ...,415,72,4,51.2615,51.4800,10.3000,12.0900,0.8472,131.9564,0,72,0.0000,0.1181
3,3,Design a cylindrical sensor housing with a diameter of 50 mm and a height of 60 mm. The wall thickness should be 2.5...,399,76,5,31.9904,66.4000,7.7200,9.7300,0.6711,45.4368,0,76,0.0000,0.3600
4,4,Create a sensor housing with dimensions of 90×70×25 mm and a wall thickness of 2 mm. This design should feature a sn...,402,74,4,41.0940,52.9800,10.2800,13.8600,0.7297,75.3320,0,74,0.0000,0.1571
5,5,"Design a waterproof sensor housing with external dimensions of 110×70×30 mm. The wall thickness should be 2.5 mm, an...",439,81,4,43.2341,65.1900,8.0400,10.8600,0.7160,81.0000,0,81,0.0000,0.3374
6,6,Create a rectangular sensor housing with external dimensions of 95×55×22 mm and a wall thickness of 1.8 mm. The lid ...,435,84,4,39.1874,61.9300,8.5000,11.3700,0.6905,84.0000,0,84,0.0000,0.2288
7,7,"Design a sensor housing with a cubic form factor, measuring 60×60×60 mm with a wall thickness of 2 mm. The lid shoul...",423,78,4,51.4055,58.4200,9.8300,12.8600,0.6795,42.4846,0,78,0.0000,0.1575
8,8,Create an oval sensor housing with major and minor axes measuring 120×60 mm and a height of 35 mm. The wall thicknes...,419,82,4,45.2971,70.8600,7.3000,9.8600,0.7561,94.1360,0,82,0.0000,0.2985
9,9,"Design a sensor housing with a trapezoidal shape, base dimensions of 100×60 mm, and a height of 40 mm. The wall thic...",403,78,4,43.8332,66.4400,8.7700,10.8200,0.6538,41.5063,0,78,0.0000,0.2972



Results saved to:
   - CSV:   G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\07_SENSOR_HOUING_PROMPT_metrics.csv
   - Excel: G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\07_SENSOR_HOUING_PROMPT_metrics.xlsx


SWITCH BOX PROMPT EVALUATION

In [15]:
# ---- Install dependencies (run once)
# !pip install torch transformers textstat pandas numpy tqdm sentencepiece language-tool-python sentence-transformers scikit-learn openpyxl

import os
import re
import io
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import textstat
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import language_tool_python
from sentence_transformers import SentenceTransformer

# ============================================================
# ---------------- Configuration -----------------------------
# ============================================================

CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "lm_model_name": "gpt2",
    "max_length": 1024,
    "stride": 512,
    "coherence_model": "sentence-transformers/all-MiniLM-L6-v2",
    "input_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\08_SWITCH_BOX_PROMPTS.csv",
    "output_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\08_SWITCH_BOX_PROMPTS_metrics.csv",
    "output_excel": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\08_SWITCH_BOX_PROMPTS_metrics.xlsx",

    # 👇 Correct public API host (no trailing slash needed, but allowed)
    "lt_api_url": "https://api.languagetool.org/v2",
    "lt_max_retries": 5,
    "lt_retry_delay": 1.0,
    "lt_request_delay": 0.2,  # ~5 req/sec — safer for public API
}

# Speed up CPU runs a bit
if CONFIG["device"] == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() - 1))

# ============================================================
# ---------------- Utility Functions -------------------------
# ============================================================

_word_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")

def simple_word_tokenize(text: str):
    return _word_re.findall(text.lower())

def sentence_split(text: str):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sents if s.strip()]

def ttr(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def mtld(tokens, ttr_threshold=0.72, min_segment=10):
    def _mtld_calc(seq):
        factors, types, token_count = 0, set(), 0
        for tok in seq:
            token_count += 1
            types.add(tok)
            if token_count >= min_segment:
                if (len(types) / token_count) <= ttr_threshold:
                    factors += 1
                    types, token_count = set(), 0
        if token_count != 0:
            # handle final partial factor
            denom = max(1 - ttr_threshold, 1e-8)
            factors += (1 - ((len(types) / max(1, token_count)) - ttr_threshold) / denom)
        return len(seq) / max(factors, 1e-8)

    if len(tokens) < min_segment:
        return float("nan")
    forward = _mtld_calc(tokens)
    reverse = _mtld_calc(list(reversed(tokens)))
    return (forward + reverse) / 2.0

@torch.no_grad()
def perplexity(text, model, tokenizer, max_length=1024, stride=512, device="cpu"):
    if not text.strip():
        return float("nan")
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = enc.input_ids.to(device)
    nlls, seq_len = [], input_ids.size(1)
    for i in range(0, seq_len, stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, seq_len)
        trg_len = end_loc - i
        input_ids_slice = input_ids[:, begin_loc:end_loc]
        target_ids = input_ids_slice.clone()
        target_ids[:, :-trg_len] = -100
        outputs = model(input_ids=input_ids_slice, labels=target_ids)
        nlls.append(outputs.loss * trg_len)
        if end_loc == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / seq_len).item()
    return float(ppl)

def safe_grammar_check(text, tool, max_retries=5, retry_delay=1.0, request_delay=0.2):
    """Returns (num_errors, num_tokens, error_rate). If tool is None or fails, returns NaNs gracefully."""
    tokens = simple_word_tokenize(text) if isinstance(text, str) else []
    if not tokens or tool is None:
        return (0, len(tokens), float("nan"))

    for attempt in range(max_retries):
        try:
            time.sleep(request_delay)  # be nice to the public API
            matches = tool.check(text)
            num_errors = sum(1 for m in matches if m.ruleId != "WHITESPACE_RULE")
            ger = num_errors / len(tokens) if len(tokens) else float("nan")
            return (num_errors, len(tokens), ger)
        except language_tool_python.RateLimitError:
            backoff = retry_delay * (attempt + 1)
            print(f"⚠️ LanguageTool rate-limited. Retrying in {backoff:.1f}s...")
            time.sleep(backoff)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Grammar check failed after {max_retries} tries: {e}")
                return (0, len(tokens), float("nan"))
            time.sleep(retry_delay * (attempt + 1))
    return (0, len(tokens), float("nan"))

def coherence_adjacent_cosine(text, embedder):
    sents = sentence_split(text)
    sents = [s for s in sents if simple_word_tokenize(s)]
    if len(sents) < 2:
        return float("nan")
    embs = embedder.encode(sents, normalize_embeddings=True, show_progress_bar=False)
    sims = [(embs[i] @ embs[i+1]) for i in range(len(embs)-1)]
    return float(np.mean(sims)) if sims else float("nan")

# -------- Robust CSV reader (fixes UnicodeDecodeError) --------
def read_csv_robust(path, expected_col="prompt"):
    """
    Try common encodings and separators; return a DataFrame and print what worked.
    Prioritizes finding the expected column name.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input CSV not found: {path}")

    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1", "iso-8859-1", "utf-16", "utf-16le", "utf-16be"]
    seps = [",", "\t", ";", "|", None]  # None -> infer (engine='python')

    last_err = None
    for enc in encodings:
        for sep in seps:
            try:
                kwargs = {"encoding": enc}
                if sep is None:
                    kwargs["sep"] = None
                    kwargs["engine"] = "python"
                else:
                    kwargs["sep"] = sep
                df = pd.read_csv(path, **kwargs)
                # success criteria: column present OR it's the only column (Excel sometimes dumps everything into one)
                if expected_col in df.columns or df.shape[1] >= 1:
                    print(f"✅ Loaded CSV with encoding='{enc}' sep='{sep or 'infer'}'  | columns={list(df.columns)}")
                    return df
            except UnicodeDecodeError as e:
                last_err = e
                continue
            except Exception as e:
                # other parser issues; keep searching
                last_err = e
                continue

    # Last resort: decode bytes with latin-1 (never fails) and let pandas parse
    try:
        with open(path, "rb") as f:
            raw = f.read()
        text = raw.decode("latin-1", errors="replace")
        df = pd.read_csv(io.StringIO(text))
        print("⚠️ Fallback path used: decoded as latin-1 with replacement; check for garbled characters.")
        return df
    except Exception as e:
        raise RuntimeError(f"Could not read CSV via robust loader. Last error: {last_err or e}")

# ============================================================
# --------------------- Main Pipeline ------------------------
# ============================================================

print(f"Using device: {CONFIG['device']}")

# 1) Load language + coherence models
tokenizer = AutoTokenizer.from_pretrained(CONFIG["lm_model_name"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["lm_model_name"]).to(CONFIG["device"]).eval()

# LanguageTool: use correct public API host; fall back to None if it fails
lt_tool = None
try:
    lt_tool = language_tool_python.LanguageTool('en-US', remote_server=CONFIG["lt_api_url"])
    # quick ping to lock in language list (avoids late failure)
    _ = lt_tool._get_languages()
    print("✅ LanguageTool connected via public API.")
except Exception as e:
    lt_tool = None
    print(f"⚠️ LanguageTool disabled (API not reachable): {e}")

embedder = SentenceTransformer(CONFIG["coherence_model"])

# 2) Load prompts (robust CSV read)
df_in = read_csv_robust(CONFIG["input_csv"], expected_col="prompt")

if "prompt" not in df_in.columns:
    raise ValueError("CSV must contain a 'prompt' column")

prompts = df_in["prompt"].dropna().astype(str).tolist()
print(f"✅ Loaded {len(prompts)} prompts.")

# 3) Score prompts
rows = []
for idx, prompt in enumerate(tqdm(prompts, desc="Scoring prompts")):
    tokens = simple_word_tokenize(prompt)
    num_chars = len(prompt)
    num_words = len(tokens)
    num_sents = len(sentence_split(prompt))

    ppl = perplexity(prompt, model, tokenizer, CONFIG["max_length"], CONFIG["stride"], CONFIG["device"])
    fre = textstat.flesch_reading_ease(prompt)
    fkgl = textstat.flesch_kincaid_grade(prompt)
    fog = textstat.gunning_fog(prompt)
    ttr_score = ttr(tokens)
    mtld_score = mtld(tokens)
    err_count, tok_count, ger = safe_grammar_check(
        prompt, lt_tool,
        max_retries=CONFIG["lt_max_retries"],
        retry_delay=CONFIG["lt_retry_delay"],
        request_delay=CONFIG["lt_request_delay"]
    )
    coh = coherence_adjacent_cosine(prompt, embedder)

    rows.append({
        "id": idx,
        "prompt": prompt,
        "chars": num_chars,
        "words": num_words,
        "sentences": num_sents,
        "perplexity": round(ppl, 4) if not pd.isna(ppl) else np.nan,
        "readability_fre": round(fre, 2),
        "readability_fkgl": round(fkgl, 2),
        "readability_fog": round(fog, 2),
        "lexdiv_ttr": round(ttr_score, 4),
        "lexdiv_mtld": round(mtld_score, 4) if not pd.isna(mtld_score) else np.nan,
        "grammar_errors": err_count,
        "grammar_tokens": tok_count,
        "grammar_error_rate": round(ger, 4) if not pd.isna(ger) else np.nan,
        "coherence_adjacent_cos": round(coh, 4) if not pd.isna(coh) else np.nan
    })

# 4) Save results
df_out = pd.DataFrame(rows)

# Jupyter-friendly display (won't crash in scripts)
try:
    pd.set_option("display.max_colwidth", 120)
    pd.set_option("display.float_format", "{:.4f}".format)
    from IPython.display import display
    display(df_out.head(10))
except Exception:
    pass

# 👉 Save CSV as UTF-8 with BOM for Windows Excel friendliness
df_out.to_csv(CONFIG["output_csv"], index=False, encoding='utf-8-sig')
df_out.to_excel(CONFIG["output_excel"], index=False, engine='openpyxl')

print("\nResults saved to:")
print(f"   - CSV:   {CONFIG['output_csv']}")
print(f"   - Excel: {CONFIG['output_excel']}")


Using device: cpu
✅ LanguageTool connected via public API.
✅ Loaded CSV with encoding='cp1252' sep=','  | columns=['prompt']
✅ Loaded 20 prompts.


Scoring prompts: 100%|██████████| 20/20 [00:28<00:00,  1.40s/it]


,id,prompt,chars,words,sentences,perplexity,readability_fre,readability_fkgl,readability_fog,lexdiv_ttr,lexdiv_mtld,grammar_errors,grammar_tokens,grammar_error_rate,coherence_adjacent_cos
0,0,Design a switch box with external dimensions of 100Ã—60Ã—40 mm. The enclosure should have a wall thickness of 2 mm a...,592,112,5,39.1458,64.1400,9.6300,12.6700,0.6875,92.8219,0,112,0.0000,0.3333
1,1,Create a rectangular switch box measuring 120Ã—80Ã—50 mm with a wall thickness of 2.5 mm. The top cover should be at...,542,103,5,54.9792,62.8300,9.4100,11.5600,0.6990,85.9244,0,103,0.0000,0.2138
2,2,Design a compact switch box with external dimensions of 80Ã—50Ã—30 mm and a wall thickness of 1.5 mm. The box must i...,537,105,6,39.3073,67.2500,8.0200,10.2700,0.6952,78.4783,0,105,0.0000,0.2043
3,3,Create a custom switch box with dimensions 150Ã—100Ã—60 mm and a robust wall thickness of 3 mm. The top should have ...,540,102,6,47.8356,72.8500,7.1600,10.2100,0.6863,80.1352,0,102,0.0000,0.3318
4,4,"Design a switch box with compact dimensions of 90Ã—70Ã—40 mm, featuring a wall thickness of 2 mm. The enclosure shou...",544,104,6,32.0905,71.3100,7.4100,10.2400,0.6827,73.3918,0,104,0.0000,0.2717
5,5,Develop a switch box with external dimensions of 110Ã—90Ã—50 mm and a wall thickness of 2.5 mm. The box should featu...,559,105,6,32.4584,65.5600,8.2600,11.0700,0.6476,62.9483,0,105,0.0000,0.2329
6,6,"Design a small switch box with dimensions of 70Ã—50Ã—35 mm, incorporating a 1.8 mm wall thickness. The enclosure sho...",533,102,6,44.1238,71.5400,6.7200,9.2500,0.6373,58.3893,0,102,0.0000,0.2733
7,7,Create a robust switch box with dimensions of 130Ã—80Ã—55 mm and a wall thickness of 2.7 mm. The enclosure should fe...,602,112,6,41.9766,68.5500,8.1300,9.7500,0.7232,112.0000,0,112,0.0000,0.2684
8,8,Design a rectangular switch box with external dimensions of 100Ã—60Ã—45 mm and a wall thickness of 2 mm. The design ...,543,101,6,36.3705,66.8100,8.0000,10.2100,0.6634,61.0895,0,101,0.0000,0.2754
9,9,Create a switch box with overall dimensions of 140Ã—90Ã—35 mm and a wall thickness of 2.2 mm. The enclosure should i...,579,107,6,39.7547,67.8500,8.0600,10.7500,0.6449,62.4426,0,107,0.0000,0.2614



Results saved to:
   - CSV:   G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\08_SWITCH_BOX_PROMPTS_metrics.csv
   - Excel: G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\08_SWITCH_BOX_PROMPTS_metrics.xlsx


WALL MOUNT PROMPT

In [16]:
# ---- Install dependencies (run once)
# !pip install torch transformers textstat pandas numpy tqdm sentencepiece language-tool-python sentence-transformers scikit-learn openpyxl

import os
import re
import io
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import textstat
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import language_tool_python
from sentence_transformers import SentenceTransformer

# ============================================================
# ---------------- Configuration -----------------------------
# ============================================================

CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "lm_model_name": "gpt2",
    "max_length": 1024,
    "stride": 512,
    "coherence_model": "sentence-transformers/all-MiniLM-L6-v2",
    "input_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\09_WALL_MOUNT_PROMPTS.csv",
    "output_csv": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\09_WALL_MOUNT_PROMPTS_metrics.csv",
    "output_excel": r"G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\09_WALL_MOUNT_PROMPTS_metrics.xlsx",

    # 👇 Correct public API host (no trailing slash needed, but allowed)
    "lt_api_url": "https://api.languagetool.org/v2",
    "lt_max_retries": 5,
    "lt_retry_delay": 1.0,
    "lt_request_delay": 0.2,  # ~5 req/sec — safer for public API
}

# Speed up CPU runs a bit
if CONFIG["device"] == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() - 1))

# ============================================================
# ---------------- Utility Functions -------------------------
# ============================================================

_word_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")

def simple_word_tokenize(text: str):
    return _word_re.findall(text.lower())

def sentence_split(text: str):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sents if s.strip()]

def ttr(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def mtld(tokens, ttr_threshold=0.72, min_segment=10):
    def _mtld_calc(seq):
        factors, types, token_count = 0, set(), 0
        for tok in seq:
            token_count += 1
            types.add(tok)
            if token_count >= min_segment:
                if (len(types) / token_count) <= ttr_threshold:
                    factors += 1
                    types, token_count = set(), 0
        if token_count != 0:
            # handle final partial factor
            denom = max(1 - ttr_threshold, 1e-8)
            factors += (1 - ((len(types) / max(1, token_count)) - ttr_threshold) / denom)
        return len(seq) / max(factors, 1e-8)

    if len(tokens) < min_segment:
        return float("nan")
    forward = _mtld_calc(tokens)
    reverse = _mtld_calc(list(reversed(tokens)))
    return (forward + reverse) / 2.0

@torch.no_grad()
def perplexity(text, model, tokenizer, max_length=1024, stride=512, device="cpu"):
    if not text.strip():
        return float("nan")
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = enc.input_ids.to(device)
    nlls, seq_len = [], input_ids.size(1)
    for i in range(0, seq_len, stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, seq_len)
        trg_len = end_loc - i
        input_ids_slice = input_ids[:, begin_loc:end_loc]
        target_ids = input_ids_slice.clone()
        target_ids[:, :-trg_len] = -100
        outputs = model(input_ids=input_ids_slice, labels=target_ids)
        nlls.append(outputs.loss * trg_len)
        if end_loc == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / seq_len).item()
    return float(ppl)

def safe_grammar_check(text, tool, max_retries=5, retry_delay=1.0, request_delay=0.2):
    """Returns (num_errors, num_tokens, error_rate). If tool is None or fails, returns NaNs gracefully."""
    tokens = simple_word_tokenize(text) if isinstance(text, str) else []
    if not tokens or tool is None:
        return (0, len(tokens), float("nan"))

    for attempt in range(max_retries):
        try:
            time.sleep(request_delay)  # be nice to the public API
            matches = tool.check(text)
            num_errors = sum(1 for m in matches if m.ruleId != "WHITESPACE_RULE")
            ger = num_errors / len(tokens) if len(tokens) else float("nan")
            return (num_errors, len(tokens), ger)
        except language_tool_python.RateLimitError:
            backoff = retry_delay * (attempt + 1)
            print(f"⚠️ LanguageTool rate-limited. Retrying in {backoff:.1f}s...")
            time.sleep(backoff)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Grammar check failed after {max_retries} tries: {e}")
                return (0, len(tokens), float("nan"))
            time.sleep(retry_delay * (attempt + 1))
    return (0, len(tokens), float("nan"))

def coherence_adjacent_cosine(text, embedder):
    sents = sentence_split(text)
    sents = [s for s in sents if simple_word_tokenize(s)]
    if len(sents) < 2:
        return float("nan")
    embs = embedder.encode(sents, normalize_embeddings=True, show_progress_bar=False)
    sims = [(embs[i] @ embs[i+1]) for i in range(len(embs)-1)]
    return float(np.mean(sims)) if sims else float("nan")

# -------- Robust CSV reader (fixes UnicodeDecodeError) --------
def read_csv_robust(path, expected_col="prompt"):
    """
    Try common encodings and separators; return a DataFrame and print what worked.
    Prioritizes finding the expected column name.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input CSV not found: {path}")

    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1", "iso-8859-1", "utf-16", "utf-16le", "utf-16be"]
    seps = [",", "\t", ";", "|", None]  # None -> infer (engine='python')

    last_err = None
    for enc in encodings:
        for sep in seps:
            try:
                kwargs = {"encoding": enc}
                if sep is None:
                    kwargs["sep"] = None
                    kwargs["engine"] = "python"
                else:
                    kwargs["sep"] = sep
                df = pd.read_csv(path, **kwargs)
                # success criteria: column present OR it's the only column (Excel sometimes dumps everything into one)
                if expected_col in df.columns or df.shape[1] >= 1:
                    print(f"✅ Loaded CSV with encoding='{enc}' sep='{sep or 'infer'}'  | columns={list(df.columns)}")
                    return df
            except UnicodeDecodeError as e:
                last_err = e
                continue
            except Exception as e:
                # other parser issues; keep searching
                last_err = e
                continue

    # Last resort: decode bytes with latin-1 (never fails) and let pandas parse
    try:
        with open(path, "rb") as f:
            raw = f.read()
        text = raw.decode("latin-1", errors="replace")
        df = pd.read_csv(io.StringIO(text))
        print("⚠️ Fallback path used: decoded as latin-1 with replacement; check for garbled characters.")
        return df
    except Exception as e:
        raise RuntimeError(f"Could not read CSV via robust loader. Last error: {last_err or e}")

# ============================================================
# --------------------- Main Pipeline ------------------------
# ============================================================

print(f"Using device: {CONFIG['device']}")

# 1) Load language + coherence models
tokenizer = AutoTokenizer.from_pretrained(CONFIG["lm_model_name"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["lm_model_name"]).to(CONFIG["device"]).eval()

# LanguageTool: use correct public API host; fall back to None if it fails
lt_tool = None
try:
    lt_tool = language_tool_python.LanguageTool('en-US', remote_server=CONFIG["lt_api_url"])
    # quick ping to lock in language list (avoids late failure)
    _ = lt_tool._get_languages()
    print("✅ LanguageTool connected via public API.")
except Exception as e:
    lt_tool = None
    print(f"⚠️ LanguageTool disabled (API not reachable): {e}")

embedder = SentenceTransformer(CONFIG["coherence_model"])

# 2) Load prompts (robust CSV read)
df_in = read_csv_robust(CONFIG["input_csv"], expected_col="prompt")

if "prompt" not in df_in.columns:
    raise ValueError("CSV must contain a 'prompt' column")

prompts = df_in["prompt"].dropna().astype(str).tolist()
print(f"✅ Loaded {len(prompts)} prompts.")

# 3) Score prompts
rows = []
for idx, prompt in enumerate(tqdm(prompts, desc="Scoring prompts")):
    tokens = simple_word_tokenize(prompt)
    num_chars = len(prompt)
    num_words = len(tokens)
    num_sents = len(sentence_split(prompt))

    ppl = perplexity(prompt, model, tokenizer, CONFIG["max_length"], CONFIG["stride"], CONFIG["device"])
    fre = textstat.flesch_reading_ease(prompt)
    fkgl = textstat.flesch_kincaid_grade(prompt)
    fog = textstat.gunning_fog(prompt)
    ttr_score = ttr(tokens)
    mtld_score = mtld(tokens)
    err_count, tok_count, ger = safe_grammar_check(
        prompt, lt_tool,
        max_retries=CONFIG["lt_max_retries"],
        retry_delay=CONFIG["lt_retry_delay"],
        request_delay=CONFIG["lt_request_delay"]
    )
    coh = coherence_adjacent_cosine(prompt, embedder)

    rows.append({
        "id": idx,
        "prompt": prompt,
        "chars": num_chars,
        "words": num_words,
        "sentences": num_sents,
        "perplexity": round(ppl, 4) if not pd.isna(ppl) else np.nan,
        "readability_fre": round(fre, 2),
        "readability_fkgl": round(fkgl, 2),
        "readability_fog": round(fog, 2),
        "lexdiv_ttr": round(ttr_score, 4),
        "lexdiv_mtld": round(mtld_score, 4) if not pd.isna(mtld_score) else np.nan,
        "grammar_errors": err_count,
        "grammar_tokens": tok_count,
        "grammar_error_rate": round(ger, 4) if not pd.isna(ger) else np.nan,
        "coherence_adjacent_cos": round(coh, 4) if not pd.isna(coh) else np.nan
    })

# 4) Save results
df_out = pd.DataFrame(rows)

# Jupyter-friendly display (won't crash in scripts)
try:
    pd.set_option("display.max_colwidth", 120)
    pd.set_option("display.float_format", "{:.4f}".format)
    from IPython.display import display
    display(df_out.head(10))
except Exception:
    pass

# 👉 Save CSV as UTF-8 with BOM for Windows Excel friendliness
df_out.to_csv(CONFIG["output_csv"], index=False, encoding='utf-8-sig')
df_out.to_excel(CONFIG["output_excel"], index=False, engine='openpyxl')

print("\nResults saved to:")
print(f"   - CSV:   {CONFIG['output_csv']}")
print(f"   - Excel: {CONFIG['output_excel']}")


Using device: cpu
✅ LanguageTool connected via public API.
✅ Loaded CSV with encoding='utf-8' sep=','  | columns=['prompt']
✅ Loaded 20 prompts.


Scoring prompts: 100%|██████████| 20/20 [00:20<00:00,  1.05s/it]


,id,prompt,chars,words,sentences,perplexity,readability_fre,readability_fkgl,readability_fog,lexdiv_ttr,lexdiv_mtld,grammar_errors,grammar_tokens,grammar_error_rate,coherence_adjacent_cos
0,0,"Design a 3D printable wall mount for a small speaker, featuring an overall size of 100×100×50 mm. The wall mount sho...",591,112,5,24.5738,62.9900,9.9400,12.8000,0.5446,42.4710,0,112,0.0000,0.4455
1,1,"Create a wall-mounted holder for a smartphone, measuring 80×150×25 mm. The holder should have a flat backplate with ...",504,99,4,27.0059,65.2700,10.1200,12.1000,0.6364,59.7746,0,99,0.0000,0.3752
2,2,"Design a wall mount for a bicycle helmet, with overall dimensions of 120×120×60 mm. The mount should have a 6 mm thi...",491,92,5,41.0276,70.1300,7.9500,10.3100,0.6304,49.8460,0,92,0.0000,0.4321
3,3,"Create a wall-mounted rack for kitchen utensils, measuring 300×50×30 mm. The rack should have a flat backplate with ...",464,85,5,28.2252,68.4500,7.7900,11.4400,0.6588,61.4632,0,85,0.0000,0.5338
4,4,Design a wall-mounted key holder with dimensions 150×80×20 mm. The holder should feature a 5 mm thick backplate with...,494,98,5,28.9025,77.0500,7.1800,9.6500,0.6327,56.8244,1,98,0.0102,0.4387
5,5,Create a wall-mounted coat rack with overall dimensions of 400×60×35 mm. The design should include a 6 mm thick back...,427,82,5,36.8722,75.1400,6.7000,9.8600,0.6585,54.5873,0,82,0.0000,0.3880
6,6,"Design a wall-mounted shelf for small books, with dimensions of 250×100×50 mm. The shelf should have a 5 mm thick ba...",436,88,5,26.5357,80.1000,6.3100,9.1500,0.6250,52.5274,0,88,0.0000,0.4832
7,7,"Create a wall-mounted holder for a hairdryer, measuring 120×80×40 mm. The holder should have a 6 mm thick backplate ...",488,92,5,27.7253,62.3400,8.9900,12.0600,0.6087,36.1498,0,92,0.0000,0.4262
8,8,Design a wall-mounted toothbrush holder with overall dimensions of 200×50×30 mm. The holder should include a 4 mm th...,475,87,5,23.4526,65.9000,8.2400,10.5300,0.6782,54.9505,0,87,0.0000,0.4534
9,9,"Create a wall-mounted holder for a tablet, measuring 220×150×30 mm. The holder should have a 5 mm thick backplate wi...",484,95,5,28.6487,70.4500,8.0000,9.9700,0.6316,51.6143,0,95,0.0000,0.3982



Results saved to:
   - CSV:   G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\09_WALL_MOUNT_PROMPTS_metrics.csv
   - Excel: G:\MAS_CAD\SYNTHETIC_DOMAIN_SPECIFIC_PROMPTS\CSV\09_WALL_MOUNT_PROMPTS_metrics.xlsx
